# S10 - Series de tiempo e inferencia en streaming

Continúa el Data Lake Gold de S04 (`campo_electrico_particionado`): hoy la pregunta cambia de "¿qué otras variables explican `CE` en el mismo instante?" (S04) a "¿el historial de `CE` predice su propio futuro?" (S10).

## Fase 1 — Business Understanding

**Objetivo:** estimar `CE` en el minuto siguiente (`label = CE_t+1`) a partir de las 8 variables físicas medidas en el minuto actual (`CE_t, CM_t, TempOut_t, OutHum_t, WindSpeed_t, Bar_t, SolarRad_t, UVIndex_t`), reutilizando el Data Lake Gold de S04 — sin reentrenar la integración ni la limpieza, ya resueltas ahí.

**Alcance:** comparar un modelo de regresión lineal y un Random Forest, ambos entrenados sobre el mismo dataset con división cronológica, contra un baseline de persistencia (`CE_t+1 ≈ CE_t`). El ganador se guarda como `PipelineModel` y se aplica después sobre lecturas que llegan en tiempo real por Kafka.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion10-series-tiempo-inferencia")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1c2327e3-eaa8-4810-8f2d-342a8f8fa9fb;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central


	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 561ms :: artifacts dl 15ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 from central in [default]
	or

:: retrieving :: org.apache.spark#spark-submit-parent-1c2327e3-eaa8-4810-8f2d-342a8f8fa9fb
	confs: [default]
	0 artifacts copied, 11 already retrieved (0kB/8ms)


26/10/04 04:25:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/04 04:25:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Fase 2 — Data Understanding

### 2.1 Cargar el Data Lake Gold de S04

`campo_electrico_particionado/` ya viene integrado, sin el código de error `CM=99999`, sin nulos y sin `DateTime` duplicada (S04, 3.3.2) — nada de eso se repite hoy.

In [2]:
RUTA_GOLD_S04 = "../s04-ml-distribuido-regresion/artifacts/campo_electrico_particionado"
VARIABLES_8 = ["CE", "CM", "TempOut", "OutHum", "WindSpeed", "Bar", "SolarRad", "UVIndex"]

df_gold = spark.read.parquet(RUTA_GOLD_S04)
print(f"Filas del Gold de S04: {df_gold.count():,}")
df_gold.orderBy("DateTime").show(5)

Filas del Gold de S04: 92,847


+-------------------+-----+-------+-------+------+---------+-----+--------+-------+-------+
|           DateTime|   CE|     CM|TempOut|OutHum|WindSpeed|  Bar|SolarRad|UVIndex|AnioMes|
+-------------------+-----+-------+-------+------+---------+-----+--------+-------+-------+
|2026-05-10 00:00:00|-0.28|24194.3|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|2026-05|
|2026-05-10 00:01:00|-0.34|24193.8|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|2026-05|
|2026-05-10 00:02:00|-0.39|24193.6|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|2026-05|
|2026-05-10 00:03:00|-0.43|24194.4|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|2026-05|
|2026-05-10 00:04:00|-0.38|24195.2|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|2026-05|
+-------------------+-----+-------+-------+------+---------+-----+--------+-------+-------+
only showing top 5 rows


## Fase 3 — Data Preparation

### 3.1 Construir el objetivo futuro (`label`)

`lead("CE", 1)` sobre el orden cronológico trae, en la fila del minuto `t`, el valor de `CE` observado en `t+1`. La última fila de la tabla no tiene un "minuto siguiente" — `na.drop()` la descarta junto con cualquier fila con un predictor nulo.

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

orden_temporal = Window.orderBy("DateTime")

dataset_ml = (
    df_gold
    .withColumn("label", F.lead("CE", 1).over(orden_temporal))
    .select("DateTime", *VARIABLES_8, "label")
    .na.drop()
)

print(f"Filas del Gold: {df_gold.count():,}")
print(f"Filas del dataset supervisado (una menos, por el ultimo lead sin futuro): {dataset_ml.count():,}")
dataset_ml.show(5)

Filas del Gold: 92,847


26/10/04 04:25:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Filas del dataset supervisado (una menos, por el ultimo lead sin futuro): 92,846


26/10/04 04:25:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------------+-----+-------+-------+------+---------+-----+--------+-------+-----+
|           DateTime|   CE|     CM|TempOut|OutHum|WindSpeed|  Bar|SolarRad|UVIndex|label|
+-------------------+-----+-------+-------+------+---------+-----+--------+-------+-----+
|2026-05-10 00:00:00|-0.28|24194.3|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|-0.34|
|2026-05-10 00:01:00|-0.34|24193.8|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|-0.39|
|2026-05-10 00:02:00|-0.39|24193.6|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|-0.43|
|2026-05-10 00:03:00|-0.43|24194.4|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|-0.38|
|2026-05-10 00:04:00|-0.38|24195.2|   16.7|  86.0|      0.0|947.3|     0.0|    0.0|-0.35|
+-------------------+-----+-------+-------+------+---------+-----+--------+-------+-----+
only showing top 5 rows


### 3.2 División cronológica 80/20

Igual que S04 anticipó (Tabla 7): el orden de las filas es el dato. `randomSplit` filtraría al azar, mezclando minutos futuros dentro del entrenamiento — por eso se numeran las filas por su orden temporal y se corta por posición, no al azar.

In [4]:
numerado = dataset_ml.withColumn("numeroFila", F.row_number().over(orden_temporal))
total_filas = numerado.count()
corte = int(total_filas * 0.80)

train = numerado.filter(F.col("numeroFila") <= corte).drop("numeroFila")
test = numerado.filter(F.col("numeroFila") > corte).drop("numeroFila")

fecha_max_train = train.agg(F.max("DateTime")).first()[0]
fecha_min_test = test.agg(F.min("DateTime")).first()[0]
if fecha_max_train >= fecha_min_test:
    raise ValueError("La division temporal no respeta el orden cronologico.")

print(f"Entrenamiento: {train.count():,} filas (hasta {fecha_max_train})")
print(f"Prueba: {test.count():,} filas (desde {fecha_min_test})")

26/10/04 04:25:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


...

26/10/04 04:25:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Entrenamiento: 74,276 filas (hasta 2026-08-19 23:15:00)


26/10/04 04:25:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Prueba: 18,570 filas (desde 2026-08-19 23:16:00)


### 3.3 Baseline de persistencia

Antes de entrenar nada: ¿qué tan bueno es simplemente asumir que `CE` no cambia de un minuto a otro (`CE_t+1 ≈ CE_t`)? Ningún modelo entrenado tiene sentido reportarlo si no supera esto — el mismo principio de 1.6 de S04 (comparar contra una alternativa antes de confiar en un modelo).

In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluador_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluador_mae = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
evaluador_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

def evaluar(predicciones, nombre):
    resultado = {
        "Modelo": nombre,
        "RMSE": evaluador_rmse.evaluate(predicciones),
        "MAE": evaluador_mae.evaluate(predicciones),
        "R2": evaluador_r2.evaluate(predicciones),
    }
    print(f"{nombre}: RMSE={resultado['RMSE']:.4f}  MAE={resultado['MAE']:.4f}  R2={resultado['R2']:.4f}")
    return resultado

test_persistencia = test.withColumn("prediction", F.col("CE"))
resultado_persistencia = evaluar(test_persistencia, "Persistencia (CE_t+1 = CE_t)")

26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Persistencia (CE_t+1 = CE_t): RMSE=0.2087  MAE=0.1316  R2=0.9461


26/10/04 04:25:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## Fase 4 — Modeling

### 4.1 Regresión lineal

Mismo patrón de S04: `VectorAssembler` ensambla las 8 variables en `features`; `LinearRegression` ya estandariza internamente (`standardization=True` por defecto), así que no hace falta un `StandardScaler` aparte (S04, 2.6/3.3.6).

In [6]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

ensamblador = VectorAssembler(inputCols=VARIABLES_8, outputCol="features")
train_ml = ensamblador.transform(train)
test_ml = ensamblador.transform(test)

lr = LinearRegression(featuresCol="features", labelCol="label")
modelo_lr = lr.fit(train_ml)

print("Coeficientes (orden de VARIABLES_8):", modelo_lr.coefficients)
print("Intercepto:", modelo_lr.intercept)

predicciones_lr = modelo_lr.transform(test_ml)
resultado_lr = evaluar(predicciones_lr, "LinearRegression")

26/10/04 04:25:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:32 WARN Instrumentation: [0222c57c] regParam is zero, which might cause numerical instability and overfitting.


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


...

26/10/04 04:25:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


LinearRegression: RMSE=0.2073  MAE=0.1325  R2=0.9468


### 4.2 Random Forest

Mismo algoritmo que S04 (`RandomForestRegressor`), sin supuesto de linealidad — útil para confirmar si la relación real es, de fondo, aproximadamente lineal.

In [7]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=50, maxDepth=8, seed=42)
modelo_rf = rf.fit(train_ml)

predicciones_rf = modelo_rf.transform(test_ml)
resultado_rf = evaluar(predicciones_rf, "RandomForestRegressor")

importancias = sorted(zip(VARIABLES_8, modelo_rf.featureImportances.toArray()), key=lambda par: -par[1])
for variable, importancia in importancias:
    print(f"{variable:10s} {importancia:.4f}")

26/10/04 04:25:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:45 WARN DAGScheduler: Broadcasting large task binary with size 1040.7 KiB


26/10/04 04:25:46 WARN DAGScheduler: Broadcasting large task binary with size 1984.5 KiB


26/10/04 04:25:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


...

26/10/04 04:25:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


RandomForestRegressor: RMSE=0.2528  MAE=0.1692  R2=0.9209
CE         0.8704
CM         0.0327
TempOut    0.0317
OutHum     0.0271
WindSpeed  0.0161
UVIndex    0.0082
Bar        0.0072
SolarRad   0.0064


## Fase 5 — Evaluation

### 5.1 Comparar contra el baseline

Si un modelo no le gana a la persistencia, su complejidad adicional no se justifica (S04, Tabla 2, columna "Series de tiempo").

In [8]:
import pandas as pd

comparacion = pd.DataFrame([resultado_persistencia, resultado_lr, resultado_rf])
comparacion["MejoraRMSE_vs_persistencia_%"] = (
    100 * (resultado_persistencia["RMSE"] - comparacion["RMSE"]) / resultado_persistencia["RMSE"]
)
comparacion

,Modelo,RMSE,MAE,R2,MejoraRMSE_vs_persistencia_%
0,Persistencia (CE_t+1 = CE_t),0.208695,0.131590,0.946103,0.000000
1,LinearRegression,0.207271,0.132527,0.946836,0.682421
2,RandomForestRegressor,0.252838,0.169171,0.920891,-21.151670


## Fase 6 — Deployment

### 6.1 Guardar el modelo seleccionado

Se guarda el `Pipeline` completo (`VectorAssembler` + modelo ganador), no solo el estimador — así la inferencia de la siguiente sección no tiene que repetir el ensamblado a mano.

In [9]:
from pyspark.ml import Pipeline, PipelineModel

RUTA_MODELO = "./artifacts/models/modelo_ce_lr"

pipeline_ganador = Pipeline(stages=[VectorAssembler(inputCols=VARIABLES_8, outputCol="features"), LinearRegression(featuresCol="features", labelCol="label")])
modelo_final = pipeline_ganador.fit(train)
modelo_final.write().overwrite().save(RUTA_MODELO)

modelo_cargado = PipelineModel.load(RUTA_MODELO)
print("Modelo guardado y recargado en:", RUTA_MODELO)
modelo_cargado.transform(test).select(*VARIABLES_8, "label", "prediction").show(5)

26/10/04 04:25:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 0

26/10/04 04:25:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:54 WARN Instrumentation: [ce837d33] regParam is zero, which might cause numerical instability and overfitting.


26/10/04 04:25:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/04 04:25:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:25:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Modelo guardado y recargado en: ./artifacts/models/modelo_ce_lr


26/10/04 04:26:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:26:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:26:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:26:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:26:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----+-------+-------+------+---------+-----+--------+-------+-----+-------------------+
|  CE|     CM|TempOut|OutHum|WindSpeed|  Bar|SolarRad|UVIndex|label|         prediction|
+----+-------+-------+------+---------+-----+--------+-------+-----+-------------------+
|0.13|24262.7|   13.3|  91.0|      0.0|951.0|     0.0|    0.0| 0.14|0.10113907621375495|
|0.14|24261.3|   13.3|  91.0|      0.0|951.0|     0.0|    0.0| 0.15|0.11102738270119983|
|0.15|24261.9|   13.3|  91.0|      0.0|951.0|     0.0|    0.0| 0.13|0.12077720402851799|
|0.13|24261.0|   13.3|  91.0|      0.0|951.0|     0.0|    0.0| 0.11|0.10125678859986231|
|0.11|24260.7|   13.3|  91.0|      0.0|950.8|     0.0|    0.0|  0.1|0.08157977948957829|
+----+-------+-------+------+---------+-----+--------+-------+-----+-------------------+
only showing top 5 rows


26/10/04 04:26:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:26:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:26:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/04 04:26:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## Inferencia en tiempo real

El modelo guardado solo necesita la **última medición** para predecir — no mantiene historial ni estado entre lecturas (confirmado en 4.1/4.2: las 8 variables ya son del mismo instante `t`). Eso simplifica mucho la inferencia en streaming: cada evento nuevo que llega por Kafka se transforma directo con `modelo_cargado.transform()`, sin ventanas ni agregaciones con estado.

Antes de correr la siguiente celda, levantar el productor sintético en otra terminal:

```bash
cd uso-campo-electrico
docker compose up -d
docker exec -it lambda26-uso-campo-electrico python producer_campo_electrico.py
```

In [10]:
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    .add("estacionId", StringType())
    .add("CE", DoubleType())
    .add("CM", DoubleType())
    .add("TempOut", DoubleType())
    .add("OutHum", DoubleType())
    .add("WindSpeed", DoubleType())
    .add("Bar", DoubleType())
    .add("SolarRad", DoubleType())
    .add("UVIndex", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "campo-electrico-eventos")
    .option("startingOffsets", "latest")
    .load()
)

eventos = (
    crudo.select(F.col("value").cast("string").alias("json_str"))
    .select(F.from_json(F.col("json_str"), esquema_evento).alias("e"))
    .select("e.*")
)

predicciones_stream = modelo_cargado.transform(eventos)

In [11]:
RUTA_PREDICCIONES_STREAM = "./artifacts/predicciones_stream"
CHECKPOINT_STREAM = "./artifacts/chk_predicciones_stream"

def guardar_predicciones_microlote(df_microlote, id_lote):
    filas = df_microlote.count()
    if filas == 0:
        print(f"Micro-lote {id_lote}: sin filas")
        return
    df_microlote.select(
        F.lit(id_lote).alias("batchId"), "estacionId", "timestamp", "CE",
        F.col("prediction").alias("prediccionSiguienteCE"),
    ).write.mode("append").parquet(RUTA_PREDICCIONES_STREAM)
    print(f"Micro-lote {id_lote}: {filas} predicciones guardadas")

# Momento 1: arrancar la consulta
consulta_predicciones = (
    predicciones_stream.writeStream
    .foreachBatch(guardar_predicciones_microlote)
    .option("checkpointLocation", CHECKPOINT_STREAM)
    .start()
)

26/10/04 04:26:09 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Dejarla correr un par de minutos mientras el productor sigue emitiendo lecturas. Cuando se haya visto suficiente, detenerla:

In [13]:
# Momento 2: detener la consulta
consulta_predicciones.stop()

spark.read.parquet(RUTA_PREDICCIONES_STREAM).orderBy(F.desc("batchId")).show(20, truncate=False)

26/10/04 04:26:35 ERROR FileFormatWriter: Aborting job 5b601fcf-08a3-4956-b89e-de9d4c811540.
java.lang.InterruptedException
	at java.base/java.util.concurrent.locks.AbstractQueuedSynchronizer.acquireSharedInterruptibly(AbstractQueuedSynchronizer.java:1134)
	at scala.concurrent.impl.Promise$DefaultPromise.tryAwait0(Promise.scala:253)
	at scala.concurrent.impl.Promise$DefaultPromise.ready(Promise.scala:265)
	at scala.concurrent.impl.Promise$DefaultPromise.ready(Promise.scala:104)
	at org.apache.spark.util.ThreadUtils$.awaitReady(ThreadUtils.scala:417)
	at org.apache.spark.scheduler.DAGScheduler.runJob(DAGScheduler.scala:1052)
	at org.apache.spark.SparkContext.runJob(SparkContext.scala:2496)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$executeWrite$4(FileFormatWriter.scala:318)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.writeAndCommit(FileFormatWriter.scala:284)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeWrite

26/10/04 04:26:35 ERROR Utils: Aborting task
org.apache.spark.TaskKilledException: Stage cancelled: [SPARK_JOB_CANCELLED] Job 94 cancelled Query [id = 656471aa-512b-40af-abdb-9d28ec2a1985, runId = f4f48afe-9f52-4d6a-9f7c-dcc2647b5fd0] was stopped SQLSTATE: XXKDA
	at org.apache.spark.TaskContextImpl.killTaskIfInterrupted(TaskContextImpl.scala:342)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:36)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:610)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:44)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory

Exception in thread "stream execution thread for [id = 656471aa-512b-40af-abdb-9d28ec2a1985, runId = f4f48afe-9f52-4d6a-9f7c-dcc2647b5fd0]" java.lang.StackOverflowError
	at java.base/java.util.regex.IntHashSet.contains(IntHashSet.java:47)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5074)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:4110)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:4110)
	at java.base/java.util.regex.Patter

+-------+--------------+-------------+-----+---------------------+
|batchId|estacionId    |timestamp    |CE   |prediccionSiguienteCE|
+-------+--------------+-------------+-----+---------------------+
|6      |estacion-ns-01|1791087988814|-6.22|-6.135440718291591   |
|5      |estacion-ns-01|1791087985810|-6.19|-6.105953144703714   |
|4      |estacion-ns-01|1791087982797|-6.2 |-6.116012633483965   |
|3      |estacion-ns-01|1791087979793|-6.16|-6.077153993616036   |
|2      |estacion-ns-01|1791087976789|-6.14|-6.057423124323649   |
|1      |estacion-ns-01|1791087973784|-6.16|-6.076612640535948   |
+-------+--------------+-------------+-----+---------------------+

